# Capstone — Finding Refresh Opportunities in Anonymized Search Content

This notebook is the executable companion to the deployed paper. It uses the bundled starter release and keeps claims bounded to the verified local evaluation.


## 1. Question

Can a ranked model help an editor find likely declining content for refresh review before inspecting the full inventory? The decision output is a review priority, not an automatic publishing action.


In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'outputs' / 'model_results.json').exists():
    ROOT = Path('..').resolve()
results = json.loads((ROOT / 'outputs' / 'model_results.json').read_text())
summary = json.loads((ROOT / 'outputs' / 'summary.json').read_text())
print(f"Rows scored: {results['input_rows']:,}")
print(f"Positive rate: {results['target_positive_rate']:.1%}")
print(f"Best model: {results['best_model']['name']}")


## 2. Data

The analysis uses `data/raw/content_refresh_anonymized.csv`: 30,000 rows, 44 source columns, 32 pseudonymized clients, trailing-90-day activity totals, and two 30-day comparison windows. The gated 79M-row warehouse release is not used in this final evaluation. Label-derived trend fields, pseudonymous IDs as features, provider/model identifiers, titles, URLs, domains, and raw queries are excluded.


In [ ]:
raw = pd.read_csv(ROOT / 'data' / 'raw' / 'content_refresh_anonymized.csv')
print(raw.shape)
print(raw['client_id'].nunique(), 'pseudonymized clients')
print(raw['trend_direction'].value_counts(dropna=False).to_dict())


## 3. Methodology

The baseline is `days_since_last_update >= 180` and `impressions_90d >= 500`. Candidates are trained with seed 42 using the repository pipeline, which compares logistic regression, a shallow decision tree, and a class-balanced random forest. Validation is a deterministic 20% client holdout (27,675 train rows and 2,325 test rows); trend fields and identifiers are not model features.


In [ ]:
print('Split:', results['split_strategy'])
print('Train rows:', results['train_rows'])
print('Test rows:', results['test_rows'])
print('Feature count:', results['feature_count'])
print('Baseline rule: stale content with at least 500 impressions')


## 4. Results (vs baseline)

The table below reads the committed model receipt, so the notebook reports the same split as the paper. Precision@50 is the operational metric because the first review batch is small.


In [ ]:
rows = []
for name, metrics in results['models'].items():
    rows.append({'method': name, 'roc_auc': metrics['roc_auc'], 'average_precision': metrics['average_precision'], 'precision_at_50': metrics['precision_at_50'], 'recall': metrics['recall'], 'f1': metrics['f1']})
baseline = results['baseline']
rows.append({'method': 'baseline_rules', 'roc_auc': baseline['baseline_roc_auc'], 'average_precision': baseline['baseline_average_precision'], 'precision_at_50': baseline['baseline_precision_at_50'], 'recall': None, 'f1': None})
pd.DataFrame(rows).round(3)


## 5. Limitations

This is retrospective decision support, not a causal experiment. The decline label is a directional proxy, the sample is one anonymized release, and the score estimates review priority rather than expected lift from an edit. A future prospective test should log interventions and comparable follow-up windows.


## 6. Ranked recommendations

1. Review high-confidence visible pages first.
2. Treat low CTR as a diagnostic candidate, not an automatic rewrite.
3. Separate engagement review from search review.
4. Monitor low-confidence items instead of spending equal effort everywhere.
5. Log outcomes to support a future prospective evaluation.


## 7. Artifacts the paper embeds

The pipeline writes `outputs/charts/`, `outputs/model_results.json`, `outputs/summary.json`, and `outputs/model_report.md`. The deployed page embeds the headline comparison, feature interpretation, and queue mix. The reproducibility command is `pip install -r requirements.txt` followed by `python scripts/run_all.py`.


## Acknowledgments & data credit

Built on the [FlyRank ML Internship dataset](https://flyrank.ai).

## Self-check

- [x] Abstract and all required paper sections are present.
- [x] Claims are bounded to the bundled anonymized starter release.
- [x] No client names, private queries, titles, URLs, or domains are exposed.
- [x] The deployed URL is recorded in `submission/paper_url.txt`.
